# `train_grpo_smoke.ipynb` — syntax & environment smoke test

Companion to `train_grpo.ipynb`. **Fast** (~1–2 min): checks imports, repo layout, `TASK_HORIZON`, and one short env run.

Run **all cells top to bottom** in Colab or locally before starting the full training notebook.

In [ ]:
# Cell 1: Minimal deps (quoted versions for zsh / shell safety)
!pip install -q pydantic httpx
!pip install -q "openenv-core[core]>=0.2.2"

In [ ]:
# Cell 2: Repo path (same logic as main notebook)
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_BRANCH = "hack1"
REPO_URL = "https://github.com/VaibhavKhandare/viral-posts-env.git"
COLAB_REPO = Path("/content/viral-posts-env")


def _is_repo_root(p: Path) -> bool:
    return (p / "server" / "viraltest_environment.py").is_file() and (p / "models.py").is_file()


def _find_local_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, here.parent, here.parent.parent):
        if _is_repo_root(cand):
            return cand
    raise FileNotFoundError(
        "Could not find project root. cd into viral-posts-env or use Colab."
    )


if Path("/content").is_dir():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO, ignore_errors=True)
    p = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(COLAB_REPO)],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{p.stderr}")
    os.chdir(COLAB_REPO)
    print("Mode: Colab")
else:
    os.chdir(_find_local_root())
    print("Mode: local")

REPO_DIR = str(Path.cwd().resolve())
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)

In [ ]:
# Cell 3: Core imports + TASK_HORIZON check
import os
import sys
from pathlib import Path

if not Path("server/viraltest_environment.py").is_file():
    for cand in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (cand / "server" / "viraltest_environment.py").is_file():
            os.chdir(cand)
            s = str(cand.resolve())
            if s not in sys.path:
                sys.path.insert(0, s)
            print("Auto chdir:", s)
            break
    else:
        raise RuntimeError("Run Cell 2 first or open from repo root.")

from models import ScheduledAction, ToolCall, ViraltestAction
from server.viraltest_environment import (
    ViraltestEnvironment,
    TAG_POOL,
    TASK_HORIZON,
    TOPIC_CATEGORIES,
)

assert TASK_HORIZON == 15, f"Expected TASK_HORIZON=15, got {TASK_HORIZON}"
print("OK: TASK_HORIZON =", TASK_HORIZON)
print("OK: tags =", len(TAG_POOL), "niches =", len(TOPIC_CATEGORIES))

In [ ]:
# Cell 4: One minimal episode (syntax + env wiring)
import random

_rng = random.Random(42)


def plan_minimal(obs_dict, day):
    topics = [t for topics in TOPIC_CATEGORIES.values() for t in topics]
    topic = topics[day % len(topics)]
    tags = [TAG_POOL[i % len(TAG_POOL)] for i in range(day, day + 3)]
    return ViraltestAction(
        scheduled_actions=[
            ScheduledAction(
                hour=12,
                action_type="post",
                content_type="carousel",
                topic=topic,
                tags=tags,
                intent="save_bait",
            )
        ]
    )


def run_episode(task, plan_fn, seed=42):
    env = ViraltestEnvironment()
    obs = env.reset(task=task, seed=seed)
    obs_dict = obs.model_dump()
    rewards = []
    for day in range(1, TASK_HORIZON + 1):
        obs = env.step(plan_fn(obs_dict, day))
        obs_dict = obs.model_dump()
        rewards.append(obs.reward or 0.0)
        if obs.done:
            break
    gs = (obs.metadata or {}).get("grader_score", 0.0)
    return {"steps": len(rewards), "total_reward": sum(rewards), "grader_score": gs}


r = run_episode("monthly_engage", plan_minimal, seed=42)
print("Episode result:", r)
assert r["steps"] == TASK_HORIZON, f"Expected {TASK_HORIZON} steps, got {r['steps']}"
print("OK: full episode completed")

In [ ]:
# Cell 5: Optional ML stack (no model download)
mods = [
    "torch",
    "transformers",
    "peft",
    "trl",
    "datasets",
    "accelerate",
]
for m in mods:
    try:
        __import__(m)
        print("OK import:", m)
    except ImportError as e:
        print("MISSING (install in full notebook):", m, "—", e)

If all cells pass, open `train_grpo.ipynb` and run the full pipeline.